In [ ]:
import os, gc, time, numpy as np, pandas as pd, torch, timm, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import rankdata
from tqdm import tqdm

LABELS = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
L = len(LABELS)
MODEL, EPOCHS, SEEDS, BS, LR = "densenet121", 7, [0, 1, 2], 64, 1e-4
OUT = r"G:\runs"; RD = rf"{OUT}\nih_train"; CD = rf"{OUT}\controls"; os.makedirs(CD, exist_ok=True)

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
    "lines.linewidth": 1.0, "grid.linewidth": 0.4, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16
def save_fig(fig, name):
    fig.savefig(rf"{CD}\{name}.pdf", bbox_inches="tight")
    fig.savefig(rf"{CD}\{name}.png", dpi=600, bbox_inches="tight"); plt.show()

# ---- tables and image caches that already exist on disk ----
nih = pd.read_csv(r"G:\nih_split.csv")
tr_i = np.where(nih.split_rev.values == "train")[0]
va_i = np.where(nih.split_rev.values == "val")[0]
te_i = np.where(nih.split_rev.values == "test")[0]
off = nih.official.values == "test"; assert (te_i == np.where(off)[0]).all()
NT = len(tr_i)                                        # NIH training size (77,353)
cx = pd.read_csv(r"G:\chexpert_final_clean.csv"); cxt = cx[cx.split == "test"].reset_index(drop=True)
arr = np.memmap(r"G:\nih_224.u8", dtype=np.uint8, mode="r", shape=(len(nih), 224, 224))
XC = np.load(r"G:\cx_test_224.npy")
Yn = nih[LABELS].values.astype(int); Ynt = Yn[off]; Yc = cxt[LABELS].values.astype(int)

# ---- predictions saved by the earlier notebooks ----
ZF = {s: np.load(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}_results\preds.npz") for s in SEEDS}   # CheXpert-trained
NPF = {s: np.load(rf"{OUT}\nih_eval\seed{s}_nih_preds.npy") for s in SEEDS}               # CheXpert-trained, all NIH
ZR = {s: np.load(rf"{RD}\{MODEL}_nih_e{EPOCHS}_seed{s}_preds.npz") for s in SEEDS}       # NIH-trained
assert (ZF[0]["test_Y"] == Yc).all() and (ZR[0]["nih_test_Y"] == Ynt).all()

MEAN = torch.tensor([0.485, 0.456, 0.406], device="cuda").view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device="cuda").view(1, 3, 1, 1)
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def mean_auc(Y, P): return float(np.mean([fauc(Y[:, i], P[:, i]) for i in range(L)]))
def bce(Y, P):
    P = np.clip(P, 1e-6, 1 - 1e-6); return float(-(Y * np.log(P) + (1 - Y) * np.log(1 - P)).mean())
def gpu_batch(A, idx, flip=False):
    x = torch.from_numpy(np.array(A[idx])).cuda(non_blocking=True).float().div(255).unsqueeze(1)
    if flip:
        f = torch.rand(len(idx), device="cuda") < 0.5
        x = torch.where(f.view(-1, 1, 1, 1), x.flip(-1), x)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD
@torch.no_grad()
def predict_idx(m, A, idx, bs=256):
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        with torch.autocast("cuda"):
            o = m(gpu_batch(A, idx[i:i + bs]))
        out.append(torch.sigmoid(o.float()).cpu())
    return torch.cat(out).numpy()
def load224(p):
    with Image.open(p) as im:
        return np.asarray(im.convert("L").resize((224, 224), Image.BILINEAR))

def get_models():
    """name -> (CheXpert-test predictions per seed, NIH-test predictions per seed, home hospital)"""
    M = {"CheXpert-152k": ({s: ZF[s]["test_P"] for s in SEEDS}, {s: NPF[s][off] for s in SEEDS}, "cx"),
         "NIH-77k": ({s: ZR[s]["cx_test_P"] for s in SEEDS}, {s: ZR[s]["nih_test_P"] for s in SEEDS}, "nih")}
    fs = {s: rf"{CD}\{MODEL}_cx{NT}_e{EPOCHS}_seed{s}_preds.npz" for s in SEEDS}
    if all(os.path.exists(f) for f in fs.values()):
        Z = {s: np.load(f) for s, f in fs.items()}
        M["CheXpert-77k"] = ({s: Z[s]["cx_test_P"] for s in SEEDS}, {s: Z[s]["nih_test_P"] for s in SEEDS}, "cx")
    return M
def groups(p): return list(pd.Series(np.arange(len(p))).groupby(np.asarray(p)).apply(np.array).values)
print("GPU:", torch.cuda.get_device_name(0), "| NIH train size:", NT, "| CheXpert test:", XC.shape, "| NIH test:", len(te_i))
print("models available:", list(get_models()))

In [ ]:
AP_CODE, PA_CODE = 0, 1          # assumed coding of CheXpert's 'AP/PA' column in this mirror; check the output below
print("CheXpert test 'AP/PA' codes:"); print(cxt["AP/PA"].value_counts())
cx_view = np.where(cxt["AP/PA"].values == AP_CODE, "AP", np.where(cxt["AP/PA"].values == PA_CODE, "PA", "other"))
nih_view = nih["view"].values                               # 'PA' / 'AP' for every NIH image
print("\nCheXpert test view counts:", pd.Series(cx_view).value_counts().to_dict())
print("NIH test view counts:     ", pd.Series(nih_view[off]).value_counts().to_dict())
print("\nDisease prevalence by view (CheXpert test):"); print(pd.DataFrame(Yc, columns=LABELS).groupby(cx_view).mean().round(3))
print("\nDisease prevalence by view (NIH test):"); print(pd.DataFrame(Ynt, columns=LABELS).groupby(nih_view[off]).mean().round(3))

In [ ]:
MODELS = get_models(); names = list(MODELS); VIEWS = ["PA", "AP"]
vc, vn = cx_view, nih_view[off]
OK = {}                                    # findings with enough positives AND negatives in this view, in BOTH test sets
for v in VIEWS:
    ok = []
    for i in range(L):
        pc_, nc_ = Yc[vc == v, i].sum(), (vc == v).sum() - Yc[vc == v, i].sum()
        pn_, nn_ = Ynt[vn == v, i].sum(), (vn == v).sum() - Ynt[vn == v, i].sum()
        if min(pc_, nc_, pn_, nn_) >= 30: ok.append(i)
    OK[v] = ok; print(f"{v}: findings used = {[LABELS[i] for i in ok]}")
def mauc(Y, Ps, mask, labs, seeds):
    return np.mean([np.mean([fauc(Y[mask, i], Ps[s][mask, i]) for s in seeds]) for i in labs])

rows = []
for n in names:
    for v in VIEWS:
        mc_, mn_ = vc == v, vn == v
        rows.append({"Model": n, "View": v, "N CheXpert": int(mc_.sum()), "N NIH": int(mn_.sum()),
                     "CheXpert test": mauc(Yc, MODELS[n][0], mc_, OK[v], SEEDS), "NIH test": mauc(Ynt, MODELS[n][1], mn_, OK[v], SEEDS)})
st = pd.DataFrame(rows)
st["In-hospital"] = np.where(st.Model.map(lambda n: MODELS[n][2]) == "cx", st["CheXpert test"], st["NIH test"])
st["External"] = np.where(st.Model.map(lambda n: MODELS[n][2]) == "cx", st["NIH test"], st["CheXpert test"])
st["View-matched gap"] = st["In-hospital"] - st["External"]
display(st.round(3)); st.to_csv(rf"{CD}\table_view_stratified.csv", index=False)

In [ ]:
B = 300; rng = np.random.default_rng(1)
gc_, gn_ = groups(cxt.patient_id.values), groups(nih.patient_id.values[off])
G = {(n, v): np.zeros(B) for n in names for v in VIEWS}
for r_ in tqdm(range(B), desc="bootstrap"):
    ic = np.concatenate([gc_[k] for k in rng.integers(0, len(gc_), len(gc_))])
    iN = np.concatenate([gn_[k] for k in rng.integers(0, len(gn_), len(gn_))])
    for v in VIEWS:
        mc_ = ic[vc[ic] == v]; mn_ = iN[vn[iN] == v]
        for n in names:
            a = mauc(Yc, MODELS[n][0], mc_, OK[v], SEEDS); b_ = mauc(Ynt, MODELS[n][1], mn_, OK[v], SEEDS)
            G[(n, v)][r_] = (a - b_) if MODELS[n][2] == "cx" else (b_ - a)
rows = []
for n in names:
    for v in VIEWS:
        pt = st[(st.Model == n) & (st.View == v)]["View-matched gap"].iloc[0]; lo, hi = np.percentile(G[(n, v)], [2.5, 97.5])
        rows.append({"Model": n, "View": v, "View-matched gap": pt, "CI low": lo, "CI high": hi})
for v in VIEWS:
    for a_, b_ in [("NIH-77k", "CheXpert-152k")] + ([("NIH-77k", "CheXpert-77k")] if "CheXpert-77k" in MODELS else []):
        d = G[(a_, v)] - G[(b_, v)]; pt = st[(st.Model == a_) & (st.View == v)]["View-matched gap"].iloc[0] - st[(st.Model == b_) & (st.View == v)]["View-matched gap"].iloc[0]
        lo, hi = np.percentile(d, [2.5, 97.5]); rows.append({"Model": f"Asymmetry: {a_} − {b_}", "View": v, "View-matched gap": pt, "CI low": lo, "CI high": hi})
vg = pd.DataFrame(rows); display(vg.round(3)); vg.to_csv(rf"{CD}\table_view_matched_gaps.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.7))
for a_, v in zip(ax, VIEWS):
    s_ = st[st.View == v]; xx = np.arange(len(s_)); w = 0.38
    a_.bar(xx - w / 2, s_["In-hospital"], w, color="0.3", edgecolor="k", lw=0.4, label="In-hospital")
    a_.bar(xx + w / 2, s_["External"], w, color="0.75", edgecolor="k", lw=0.4, label="External")
    a_.set_xticks(xx); a_.set_xticklabels(s_.Model, rotation=25, ha="right"); a_.set(title=f"{v} images only", ylabel="Mean AUROC", ylim=(0.5, 0.95))
    a_.legend(frameon=False, fontsize=6); a_.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_view_matched")

In [ ]:
nh_tv = ~off; rows = []
for nm, mask in [("NIH train_val (77% of NIH)", nh_tv), ("NIH official test", off)]:
    r = {"Subset": nm, "N": int(mask.sum()), "AP share": float((nih_view[mask] == "AP").mean())}
    r["All images"] = np.mean([np.mean([fauc(Yn[mask, i], NPF[s][mask, i]) for s in SEEDS]) for i in range(L)])
    for v in VIEWS:
        mv = mask & (nih_view == v)
        labs = [i for i in range(L) if min(Yn[mv, i].sum(), mv.sum() - Yn[mv, i].sum()) >= 30]
        r[f"{v} only"] = np.mean([np.mean([fauc(Yn[mv, i], NPF[s][mv, i]) for s in SEEDS]) for i in labs])
    rows.append(r)
cm_ = pd.DataFrame(rows).set_index("Subset"); display(cm_.round(3)); cm_.to_csv(rf"{CD}\table_nih_subset_case_mix.csv")
print("CheXpert-152k model on NIH. If the 'PA only' and 'AP only' columns are similar across the two subsets,")
print("the difference in 'All images' comes from the PA/AP mix, not from the images themselves.")

In [ ]:
cxm = cx[cx.split.isin(["train", "val"])].reset_index(drop=True)          # CheXpert train + val rows
Ycm = cxm[LABELS].values.astype(int)
CXF = r"G:\cx_trainval_224.u8"
print(len(cxm), "CheXpert train+val images; needs about", round(len(cxm) * 224 * 224 / 1e9, 1), "GB")
if not os.path.exists(CXF):
    mmc = np.memmap(CXF, dtype=np.uint8, mode="w+", shape=(len(cxm), 224, 224))
    bad = []
    def put(i):
        try: mmc[i] = load224(cxm.img_path.iloc[i])
        except Exception: bad.append(i)
    with ThreadPoolExecutor(12) as pool:
        list(tqdm(pool.map(put, range(len(cxm))), total=len(cxm), desc="caching CheXpert train+val"))
    mmc.flush(); del mmc
    print("unreadable images:", len(bad))
CXA = np.memmap(CXF, dtype=np.uint8, mode="r", shape=(len(cxm), 224, 224))
cx_tr_pool = np.where(cxm.split.values == "train")[0]; cx_va = np.where(cxm.split.values == "val")[0]
print("cache ready | pool:", len(cx_tr_pool), "val:", len(cx_va))

In [ ]:
def subset_idx(seed, n_target):
    """random CheXpert training subset of exactly n_target images, chosen by patient"""
    r = np.random.default_rng(1000 + seed)
    pats = cxm.patient_id.values[cx_tr_pool]
    cnt = pd.Series(pats).value_counts().to_dict()
    up = np.unique(pats); r.shuffle(up)
    chosen, tot = set(), 0
    for p in up:
        chosen.add(p); tot += cnt[p]
        if tot >= n_target: break
    idx = cx_tr_pool[np.isin(pats, list(chosen))]
    return np.sort(r.permutation(idx)[:n_target])

lossf = nn.BCEWithLogitsLoss()
def train_cx_subset(seed, n_target=NT):
    tag = f"{MODEL}_cx{n_target}_e{EPOCHS}_seed{seed}"
    if os.path.exists(rf"{CD}\{tag}_preds.npz"):
        print(f"seed {seed}: already finished, skipping"); return
    BEST, CKPT, HIST = rf"{CD}\{tag}.pt", rf"{CD}\{tag}_last.pt", rf"{CD}\{tag}_history.csv"
    tr_idx = subset_idx(seed, n_target)
    print(f"seed {seed}: {len(tr_idx)} training images from {len(set(cxm.patient_id.values[tr_idx]))} patients")
    torch.manual_seed(seed); np.random.seed(seed)
    m = timm.create_model(MODEL, pretrained=True, num_classes=L).cuda()
    o = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    spe = int(np.ceil(len(tr_idx) / BS))
    sc = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=EPOCHS * spe)
    sl = torch.amp.GradScaler("cuda")
    hist, best, start = [], 0.0, 0
    if os.path.exists(CKPT):
        ck = torch.load(CKPT, map_location="cuda", weights_only=False)
        m.load_state_dict(ck["model"]); o.load_state_dict(ck["opt"]); sc.load_state_dict(ck["sched"])
        sl.load_state_dict(ck["scaler"]); hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1
        print(f"seed {seed}: resumed, next epoch {start + 1}")
    for ep in range(start, EPOCHS):
        perm = np.random.default_rng(seed * 100 + ep).permutation(tr_idx); m.train(); t0 = time.time()
        rl, n = torch.zeros((), device="cuda"), 0
        for k in tqdm(range(spe), desc=f"seed {seed} epoch {ep + 1}/{EPOCHS}"):
            idx = np.sort(perm[k * BS:(k + 1) * BS])
            x = gpu_batch(CXA, idx, flip=True); y = torch.from_numpy(Ycm[idx]).cuda().float()
            o.zero_grad(set_to_none=True)
            with torch.autocast("cuda"):
                loss = lossf(m(x), y)
            sl.scale(loss).backward(); sl.step(o); sl.update(); sc.step()
            rl += loss.detach() * len(idx); n += len(idx)
        Pv = predict_idx(m, CXA, cx_va); Yv = Ycm[cx_va]
        row = dict(epoch=ep + 1, train_loss=float(rl) / n, val_loss=bce(Yv, Pv), val_auroc=mean_auc(Yv, Pv), sec=time.time() - t0)
        hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"seed {seed} ep {ep + 1} | loss {row['train_loss']:.4f}/{row['val_loss']:.4f} | val AUROC {row['val_auroc']:.4f} | {row['sec'] / 60:.1f} min")
        if row["val_auroc"] > best:
            best = row["val_auroc"]; torch.save(m.state_dict(), BEST)
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=sc.state_dict(), scaler=sl.state_dict(),
                        hist=hist, best=best, ep=ep), CKPT)
    m.load_state_dict(torch.load(BEST, map_location="cuda", weights_only=True))
    z = dict(cx_test_P=predict_idx(m, XC, np.arange(len(XC))), cx_test_Y=Yc,
             nih_test_P=predict_idx(m, arr, te_i), nih_test_Y=Ynt)
    np.savez(rf"{CD}\{tag}_preds.npz", **z)
    print(f"seed {seed}: CheXpert test AUROC {mean_auc(Yc, z['cx_test_P']):.4f} | NIH test (external) AUROC {mean_auc(Ynt, z['nih_test_P']):.4f}")
    del m, o, sc, sl; gc.collect(); torch.cuda.empty_cache()

In [ ]:
for s in SEEDS:
    train_cx_subset(s)

In [ ]:
MODELS = get_models(); assert "CheXpert-77k" in MODELS, "run Step 1C first"
names = list(MODELS)
pmf = lambda a: f"{np.mean(a):.3f} ± {np.std(a, ddof=1):.3f}"
mat = pd.DataFrame({"CheXpert test": {n: pmf([mean_auc(Yc, MODELS[n][0][s]) for s in SEEDS]) for n in names},
                    "NIH test": {n: pmf([mean_auc(Ynt, MODELS[n][1][s]) for s in SEEDS]) for n in names}})
mat.index.name = "Trained on"; display(mat); mat.to_csv(rf"{CD}\table_size_control_matrix.csv")

B = 300; gc_, gn_ = groups(cxt.patient_id.values), groups(nih.patient_id.values[off]); rng = np.random.default_rng(0)
bc = {n: np.zeros((B, L)) for n in names}; bn = {n: np.zeros((B, L)) for n in names}
for r_ in tqdm(range(B), desc="bootstrap"):
    ic = np.concatenate([gc_[k] for k in rng.integers(0, len(gc_), len(gc_))])
    iN = np.concatenate([gn_[k] for k in rng.integers(0, len(gn_), len(gn_))])
    for n in names:
        for i in range(L):
            bc[n][r_, i] = np.mean([fauc(Yc[ic, i], MODELS[n][0][s][ic, i]) for s in SEEDS])
            bn[n][r_, i] = np.mean([fauc(Ynt[iN, i], MODELS[n][1][s][iN, i]) for s in SEEDS])
pc = {n: np.array([np.mean([fauc(Yc[:, i], MODELS[n][0][s][:, i]) for s in SEEDS]) for i in range(L)]) for n in names}
pn = {n: np.array([np.mean([fauc(Ynt[:, i], MODELS[n][1][s][:, i]) for s in SEEDS]) for i in range(L)]) for n in names}
gap_pt = {n: (pc[n] - pn[n]) if MODELS[n][2] == "cx" else (pn[n] - pc[n]) for n in names}         # in-hospital minus external
gap_b = {n: (bc[n] - bn[n]) if MODELS[n][2] == "cx" else (bn[n] - bc[n]) for n in names}
ci = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
def row_of(lbl, pt, bs):
    lo, hi = ci(bs); mlo, mhi = np.percentile(bs.mean(1), [2.5, 97.5])
    r = {"Quantity": lbl}
    for i, l in enumerate(LABELS): r[l] = f"{pt[i]:+.3f} [{lo[i]:+.3f}, {hi[i]:+.3f}]"
    r["Mean"] = f"{pt.mean():+.3f} [{mlo:+.3f}, {mhi:+.3f}]"; return r
rows = [row_of(f"Gap, {n} (in-hospital − external)", gap_pt[n], gap_b[n]) for n in names]
rows.append(row_of("Size effect (CheXpert-77k gap − CheXpert-152k gap)", gap_pt["CheXpert-77k"] - gap_pt["CheXpert-152k"], gap_b["CheXpert-77k"] - gap_b["CheXpert-152k"]))
rows.append(row_of("Original asymmetry (NIH-77k gap − CheXpert-152k gap)", gap_pt["NIH-77k"] - gap_pt["CheXpert-152k"], gap_b["NIH-77k"] - gap_b["CheXpert-152k"]))
rows.append(row_of("Size-controlled asymmetry (NIH-77k gap − CheXpert-77k gap)", gap_pt["NIH-77k"] - gap_pt["CheXpert-77k"], gap_b["NIH-77k"] - gap_b["CheXpert-77k"]))
tab = pd.DataFrame(rows).set_index("Quantity"); display(tab[["Mean"]]); tab.T.to_csv(rf"{CD}\table_size_control_gaps.csv")
display(tab)

x = np.arange(L); clip = lambda a: np.clip(a, 0, None); fig, ax = plt.subplots(figsize=(COL2, 2.9)); w = 0.27
for k, (n, sh) in enumerate(zip(names, ["0.25", "0.55", "0.85"])):
    c = ci(gap_b[n]); ax.bar(x + (k - 1) * w, gap_pt[n], w, yerr=[clip(gap_pt[n] - c[0]), clip(c[1] - gap_pt[n])], capsize=1.5, color=sh, edgecolor="k", lw=0.4, label=n)
ax.axhline(0, color="k", lw=0.6); ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=25, ha="right")
ax.set(ylabel="AUROC drop (in-hospital − external)"); ax.legend(frameon=False, fontsize=6); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); save_fig(fig, "fig_size_control_gaps")

In [ ]:
# ===================== STEP 0: one fast, exact, PAIRED bootstrap engine for steps 1 and 2 =====================
import glob
NB = 2000                       # paired resamples (use 300 only for a quick test; Holm needs thousands)
class FastAUC:
    """Exact Mann-Whitney AUROC when every image carries an integer weight w = how many times its patient was drawn.
    Scores are sorted ONCE; each bootstrap replicate is then a handful of O(N) vector operations (about 10x faster than re-ranking)."""
    def __init__(self, y, s):
        o = np.argsort(s, kind="mergesort"); ss = np.asarray(s)[o]
        self.o = o; self.y = np.asarray(y)[o].astype(float); self.starts = np.flatnonzero(np.r_[True, ss[1:] != ss[:-1]])      # tie groups
    def auc(self, w):
        w = w[self.o]; gp = np.add.reduceat(w * self.y, self.starts); gn = np.add.reduceat(w * (1 - self.y), self.starts)
        P_, N_ = gp.sum(), gn.sum()
        if P_ <= 0 or N_ <= 0: return np.nan
        return float((gp * (np.cumsum(gn) - gn + 0.5 * gn)).sum() / (P_ * N_))

# ---- unit test against your own fauc on a real patient-level resample: must agree to numerical precision ----
_r = np.random.default_rng(0); _g = groups(cxt.patient_id.values); _ic = np.concatenate([_g[k] for k in _r.integers(0, len(_g), len(_g))])
_w = np.bincount(_ic, minlength=len(Yc)).astype(float)
for _i in range(L):
    _a = fauc(Yc[_ic, _i], ZF[0]["test_P"][_ic, _i]); _b = FastAUC(Yc[:, _i], ZF[0]["test_P"][:, _i]).auc(_w)
    assert abs(_a - _b) < 1e-9, (LABELS[_i], _a, _b)
print("FastAUC matches your fauc on a real patient-level resample for all", L, "findings")

def holm(p):                                                    # Holm step-down adjusted p-values
    p = np.asarray(p, float); o = np.argsort(p); out = np.empty(len(p)); run = 0.0
    for r_, i in enumerate(o): run = max(run, min(1.0, (len(p) - r_) * p[i])); out[i] = run
    return out
def boot_p(d, nb): return min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1.0 / nb)       # two-sided; the smallest attainable value is 1/nb
ci95 = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
print(f"NB = {NB}: smallest attainable p = {1 / NB:.4f}; Holm over 6 tests can reach {6 / NB:.4f}")

In [ ]:
# ===================== STEP 1A: more random 77,353-image CheXpert subsets =====================
EXTRA_SUBSETS = [3, 4, 5, 6, 7]          # about 18 min each on your RTX 4070 SUPER. Finished subsets are skipped, interrupted ones resume.
for s in EXTRA_SUBSETS:
    train_cx_subset(s)                   # saves {MODEL}_cx{NT}_e{EPOCHS}_seed{s}_preds.npz in CD

In [ ]:
# ===================== STEP 1B: load every model once and run the paired bootstrap (all strata at once) =====================
MODELS = get_models()                                           # your function: CheXpert-152k and NIH-77k predictions per seed
RUNS = []
def add(kind, s, pc, pn, home): RUNS.append(dict(id=f"{kind}_s{s}", kind=kind, seed=s, home=home, pc=pc, pn=pn))
for s in SEEDS:
    add("CX152", s, MODELS["CheXpert-152k"][0][s], MODELS["CheXpert-152k"][1][s], "cx")
    add("NIH77", s, MODELS["NIH-77k"][0][s],       MODELS["NIH-77k"][1][s],       "nih")
for f in sorted(glob.glob(os.path.join(CD, f"{MODEL}_cx{NT}_e{EPOCHS}_seed*_preds.npz"))):          # EVERY random CheXpert-77k subset found on disk
    s = int(os.path.basename(f).split("seed")[-1].split("_")[0]); z = np.load(f); add("CX77", s, z["cx_test_P"], z["nih_test_P"], "cx")
print(pd.DataFrame(RUNS).groupby("kind").seed.agg(lambda x: sorted(x)).to_string())
assert sum(r["kind"] == "CX77" for r in RUNS) >= 2, "need at least 2 CheXpert-77k subsets: run Step 1A or the training cell of your notebook"

for r in RUNS:                                                  # sort every score vector once
    r["fc"] = {i: FastAUC(Yc[:, i], r["pc"][:, i]) for i in range(L)}; r["fn"] = {i: FastAUC(Ynt[:, i], r["pn"][:, i]) for i in range(L)}
vc, vn = cx_view, nih_view[off]
STRATA = {"All": (np.ones(len(Yc), bool), np.ones(len(Ynt), bool)), "PA": (vc == "PA", vn == "PA"), "AP": (vc == "AP", vn == "AP")}
LAB_USE = {"All": list(range(L)), "PA": OK["PA"], "AP": OK["AP"]}                              # findings with enough positives AND negatives in that view, in BOTH hospitals
gc_, gn_ = groups(cxt.patient_id.values), groups(nih.patient_id.values[off])                  # the resampling unit is the PATIENT
AUCB = {(r["id"], site, st): np.full((NB, L), np.nan) for r in RUNS for site in ("cx", "nih") for st in STRATA}
AUCP = {(r["id"], site, st): np.full(L, np.nan) for r in RUNS for site in ("cx", "nih") for st in STRATA}
w1c, w1n = np.ones(len(Yc)), np.ones(len(Ynt))                                                 # point estimates on the full test sets
for st, (mc_, mn_) in STRATA.items():
    for r in RUNS:
        for i in LAB_USE[st]: AUCP[(r["id"], "cx", st)][i] = r["fc"][i].auc(w1c * mc_); AUCP[(r["id"], "nih", st)][i] = r["fn"][i].auc(w1n * mn_)
rng = np.random.default_rng(2024)
for b in tqdm(range(NB), desc="paired patient bootstrap"):
    ic = np.concatenate([gc_[k] for k in rng.integers(0, len(gc_), len(gc_))]); iN = np.concatenate([gn_[k] for k in rng.integers(0, len(gn_), len(gn_))])
    wc = np.bincount(ic, minlength=len(Yc)).astype(float); wn = np.bincount(iN, minlength=len(Ynt)).astype(float)      # the SAME sampled patients for every model, view and finding
    for st, (mc_, mn_) in STRATA.items():
        wcs, wns = wc * mc_, wn * mn_
        for r in RUNS:
            for i in LAB_USE[st]: AUCB[(r["id"], "cx", st)][b, i] = r["fc"][i].auc(wcs); AUCB[(r["id"], "nih", st)][b, i] = r["fn"][i].auc(wns)
print("bootstrap done:", len(RUNS), "model runs x 3 strata x 2 hospitals x", NB, "paired resamples")

In [ ]:
# ===================== STEP 1C: size-matched analysis, repeated random subsets, both directions =====================
RK = {k: [r for r in RUNS if r["kind"] == k] for k in ("CX152", "NIH77", "CX77")}; R = len(RK["CX77"])
macro = lambda x: np.nanmean(x, axis=-1)
def gapB(r, st): a, b = AUCB[(r["id"], "cx", st)], AUCB[(r["id"], "nih", st)]; return (a - b) if r["home"] == "cx" else (b - a)       # in-hospital minus external, (NB, L)
def gapP(r, st): a, b = AUCP[(r["id"], "cx", st)], AUCP[(r["id"], "nih", st)]; return (a - b) if r["home"] == "cx" else (b - a)       # (L,)
IDX = np.random.default_rng(7).integers(0, R, (NB, R))                                         # nested bootstrap: the R subset draws are resampled too, same draws for every quantity
dmean = lambda M: np.take_along_axis(M, IDX, 1).mean(1)
def derive(st):
    gN, gC = np.mean([gapB(r, st) for r in RK["NIH77"]], 0), np.mean([gapB(r, st) for r in RK["CX152"]], 0)           # seed-averaged, (NB, L)
    g77 = np.stack([gapB(r, st) for r in RK["CX77"]], 1)                                                              # (NB, R, L): one slice per random subset
    pN, pC = np.mean([gapP(r, st) for r in RK["NIH77"]], 0), np.mean([gapP(r, st) for r in RK["CX152"]], 0); p77 = np.stack([gapP(r, st) for r in RK["CX77"]], 0)     # (R, L)
    n77 = np.take_along_axis(g77, IDX[:, :, None], 1).mean(1)                                                         # (NB, L)
    B = {"gap NIH-77k": gN, "gap CheXpert-152k": gC, "gap CheXpert-77k": n77, "original asymmetry (NIH-77k − CheXpert-152k)": gN - gC,
         "size effect (CheXpert-77k − CheXpert-152k)": n77 - gC, "size-controlled asymmetry (NIH-77k − CheXpert-77k)": gN - n77}
    P = {"gap NIH-77k": pN, "gap CheXpert-152k": pC, "gap CheXpert-77k": p77.mean(0), "original asymmetry (NIH-77k − CheXpert-152k)": pN - pC,
         "size effect (CheXpert-77k − CheXpert-152k)": p77.mean(0) - pC, "size-controlled asymmetry (NIH-77k − CheXpert-77k)": pN - p77.mean(0)}
    return B, P, g77, p77
B_all, P_all, g77_all, p77_all = derive("All")

rows = []
for k in B_all:
    est = float(np.nanmean(P_all[k])); d = macro(B_all[k]); lo, hi = np.percentile(d, [2.5, 97.5]); rows.append(dict(quantity=k, estimate=est, lo=lo, hi=hi, p=boot_p(d, NB)))
S1 = pd.DataFrame(rows); fam = S1.quantity.str.contains("asymmetry|size effect")
S1.loc[fam, "p_holm"] = holm(S1.loc[fam, "p"].values)                                           # Holm family 1: the three decomposition tests
S1["significant"] = S1.p_holm < .05
ratio = macro(B_all["size effect (CheXpert-77k − CheXpert-152k)"]) / macro(B_all["original asymmetry (NIH-77k − CheXpert-152k)"])
ok = np.abs(macro(B_all["original asymmetry (NIH-77k − CheXpert-152k)"])) > .01
S1_frac = (float(np.nanmean(P_all["size effect (CheXpert-77k − CheXpert-152k)"]) / np.nanmean(P_all["original asymmetry (NIH-77k − CheXpert-152k)"])), *np.percentile(ratio[ok], [2.5, 97.5]))
display(S1.round(4)); S1.to_csv(os.path.join(CD, "table_size_matched_rigorous.csv"), index=False)
print(f"Share of the original asymmetry attributable to training-set size: {S1_frac[0]:.2f}  [{S1_frac[1]:.2f}, {S1_frac[2]:.2f}]   (negative or near 0 = size does not explain it)")

per = pd.DataFrame([dict(subset_seed=r["seed"], CX77_gap=float(np.nanmean(gapP(r, "All"))), size_controlled_asymmetry=float(np.nanmean(P_all["gap NIH-77k"]) - np.nanmean(gapP(r, "All")))) for r in RK["CX77"]])
print(f"\nPer random CheXpert-77k subset ({R} subsets): mean {per.size_controlled_asymmetry.mean():+.4f}, sd {per.size_controlled_asymmetry.std(ddof=1):.4f}, range [{per.size_controlled_asymmetry.min():+.4f}, {per.size_controlled_asymmetry.max():+.4f}]")
display(per.round(4)); per.to_csv(os.path.join(CD, "table_size_matched_per_subset.csv"), index=False)

gl77 = np.take_along_axis(np.stack([gapB(r, "All") for r in RK["CX77"]], 1), IDX[:, :, None], 1).mean(1); gN_ = np.mean([gapB(r, "All") for r in RK["NIH77"]], 0)
lab = pd.DataFrame({LABELS[i]: dict(estimate=float(np.nanmean(P_all["size-controlled asymmetry (NIH-77k − CheXpert-77k)"][i])), lo=np.percentile(gN_[:, i] - gl77[:, i], 2.5), hi=np.percentile(gN_[:, i] - gl77[:, i], 97.5)) for i in range(L)}).T
display(lab.round(3)); lab.to_csv(os.path.join(CD, "table_size_matched_per_finding.csv"))

fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.9), gridspec_kw=dict(width_ratios=[1, 1.15, 1.25]))
ax[0].scatter(per.size_controlled_asymmetry, range(R), color="0.3", s=14, zorder=3, label="one random subset"); r0 = S1[S1.quantity.str.startswith("size-controlled")].iloc[0]
ax[0].axvline(r0.estimate, color="k", lw=.8); ax[0].axvspan(r0.lo, r0.hi, color="0.8", alpha=.6, label="mean, 95% CI"); ax[0].axvline(0, color="r", ls="--", lw=.7)
ax[0].set_yticks(range(R)); ax[0].set_yticklabels([f"subset {s}" for s in per.subset_seed]); ax[0].set(xlabel="Size-controlled asymmetry (AUROC)", title="Repeated random 77k subsets"); ax[0].legend(frameon=False, fontsize=5.5)
for k, q in enumerate(["gap NIH-77k", "gap CheXpert-152k", "gap CheXpert-77k"]):
    s_ = S1[S1.quantity == q].iloc[0]; ax[1].bar(k, s_.estimate, .6, yerr=[[s_.estimate - s_.lo], [s_.hi - s_.estimate]], capsize=3, color=["0.25", "0.55", "0.85"][k], edgecolor="k", lw=.4)
ax[1].set_xticks(range(3)); ax[1].set_xticklabels(["NIH-77k", "CheXpert-152k", "CheXpert-77k"], rotation=20); ax[1].set(ylabel="Gap: in-hospital − external AUROC", title="Gaps"); ax[1].grid(True, axis="y", alpha=.3)
for k, q in enumerate(["original asymmetry (NIH-77k − CheXpert-152k)", "size effect (CheXpert-77k − CheXpert-152k)", "size-controlled asymmetry (NIH-77k − CheXpert-77k)"]):
    s_ = S1[S1.quantity == q].iloc[0]; ax[2].bar(k, s_.estimate, .6, yerr=[[s_.estimate - s_.lo], [s_.hi - s_.estimate]], capsize=3, color=["0.25", "0.6", "0.85"][k], edgecolor="k", lw=.4, hatch=["", "///", ""][k])
ax[2].axhline(0, color="k", lw=.6); ax[2].set_xticks(range(3)); ax[2].set_xticklabels(["original", "size effect", "size-controlled"], rotation=20); ax[2].set(title="Decomposition (original = size + size-controlled)"); ax[2].grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_size_matched_rigorous")

In [ ]:
# ===================== STEP 2: PA/AP-stratified analysis, 2,000 paired resamples, Holm per family, effect sizes =====================
D = {st: derive(st) for st in ("All", "PA", "AP")}                                              # same replicates, same patients, same subset draws for every stratum and model
rows = []
def add_row(family, quantity, st, d, est):
    d = macro(d) if d.ndim > 1 else d; lo, hi = np.percentile(d, [2.5, 97.5]); rows.append(dict(family=family, quantity=quantity, stratum=st, estimate=float(est), lo=lo, hi=hi, p=boot_p(d, NB)))
GAPQ = ["gap NIH-77k", "gap CheXpert-152k", "gap CheXpert-77k"]; ASQ = ["original asymmetry (NIH-77k − CheXpert-152k)", "size-controlled asymmetry (NIH-77k − CheXpert-77k)"]
for st in ("PA", "AP"):                                                                         # F1: is the view-matched gap different from 0?
    for q in GAPQ: add_row("F1 view-matched gaps vs 0", q, st, D[st][0][q], np.nanmean(D[st][1][q]))
for st in ("PA", "AP"):                                                                         # F2: is the asymmetry still there inside one view?
    for q in ASQ: add_row("F2 view-matched asymmetry vs 0", q, st, D[st][0][q], np.nanmean(D[st][1][q]))
for q in GAPQ + ASQ:                                                                            # F3: does the view change the gap or the asymmetry? (PA minus AP)
    add_row("F3 PA − AP contrast", q, "PA − AP", macro(D["PA"][0][q]) - macro(D["AP"][0][q]), np.nanmean(D["PA"][1][q]) - np.nanmean(D["AP"][1][q]))
S2 = pd.DataFrame(rows); S2["p_holm"] = S2.groupby("family").p.transform(lambda x: holm(x.values)); S2["significant"] = S2.p_holm < .05
print("Holm is applied inside each family (F1 6 tests, F2 4 tests, F3 5 tests); smallest attainable Holm p in F1 =", round(6 / NB, 4))
display(S2.round(4)); S2.to_csv(os.path.join(CD, "table_view_stratified_rigorous.csv"), index=False)

rows = []                                                                                       # effect size: how much of the asymmetry remains after view matching
for q in ASQ:
    for st in ("PA", "AP"):
        den = macro(D["All"][0][q]); num = macro(D[st][0][q]); ok = np.abs(den) > .01
        rat = num[ok] / den[ok]; pt = np.nanmean(D[st][1][q]) / np.nanmean(D["All"][1][q]); rows.append(dict(quantity=q, stratum=st, share_of_all_image_asymmetry_remaining=pt, lo=np.percentile(rat, 2.5), hi=np.percentile(rat, 97.5)))
EFF = pd.DataFrame(rows); display(EFF.round(3)); EFF.to_csv(os.path.join(CD, "table_view_stratified_effect_sizes.csv"), index=False)
print("A share near 1 inside BOTH views means the PA/AP mix does not explain the asymmetry; a share near 0 means it does.")

fig, axes = plt.subplots(1, 3, figsize=(COL2, 3.1))
for a_, fam_ in zip(axes, sorted(S2.family.unique())):
    g_ = S2[S2.family == fam_].reset_index(drop=True)
    for i, r in g_.iterrows():
        a_.errorbar(r.estimate, i, xerr=[[r.estimate - r.lo], [r.hi - r.estimate]], fmt="o", color="k", mfc="k" if r.significant else "w", ms=4, capsize=2, lw=.8)
    a_.axvline(0, color="r", ls="--", lw=.7); a_.set_yticks(range(len(g_))); a_.set_yticklabels([f"{q.split(' (')[0]}, {s}" for q, s in zip(g_.quantity, g_.stratum)], fontsize=5.3); a_.invert_yaxis()
    a_.set(title=fam_, xlabel="AUROC difference (95% CI)"); a_.grid(True, axis="x", alpha=.3)
fig.suptitle("Filled = significant after Holm correction within the family", fontsize=7); fig.tight_layout(); save_fig(fig, "fig_view_stratified_rigorous")

In [ ]:
# ===================== item 2, cell 1: exact, fast, PAIRED bootstrap engine =====================
import glob
NB = 2000                       # paired resamples (use 300 only for a quick test; Holm needs thousands)
class FastAUC:
    """Exact Mann-Whitney AUROC when every image carries an integer weight w = how many times its patient was drawn.
    Scores are sorted ONCE; each bootstrap replicate is then a handful of O(N) vector operations (about 10x faster than re-ranking)."""
    def __init__(self, y, s):
        o = np.argsort(s, kind="mergesort"); ss = np.asarray(s)[o]
        self.o = o; self.y = np.asarray(y)[o].astype(float); self.starts = np.flatnonzero(np.r_[True, ss[1:] != ss[:-1]])      # tie groups
    def auc(self, w):
        w = w[self.o]; gp = np.add.reduceat(w * self.y, self.starts); gn = np.add.reduceat(w * (1 - self.y), self.starts)
        P_, N_ = gp.sum(), gn.sum()
        if P_ <= 0 or N_ <= 0: return np.nan
        return float((gp * (np.cumsum(gn) - gn + 0.5 * gn)).sum() / (P_ * N_))

# ---- unit test against your own fauc on a real patient-level resample: must agree to numerical precision ----
_r = np.random.default_rng(0); _g = groups(cxt.patient_id.values); _ic = np.concatenate([_g[k] for k in _r.integers(0, len(_g), len(_g))])
_w = np.bincount(_ic, minlength=len(Yc)).astype(float)
for _i in range(L):
    _a = fauc(Yc[_ic, _i], ZF[0]["test_P"][_ic, _i]); _b = FastAUC(Yc[:, _i], ZF[0]["test_P"][:, _i]).auc(_w)
    assert abs(_a - _b) < 1e-9, (LABELS[_i], _a, _b)
print("FastAUC matches your fauc on a real patient-level resample for all", L, "findings")

def holm(p):                                                    # Holm step-down adjusted p-values
    p = np.asarray(p, float); o = np.argsort(p); out = np.empty(len(p)); run = 0.0
    for r_, i in enumerate(o): run = max(run, min(1.0, (len(p) - r_) * p[i])); out[i] = run
    return out
def boot_p(d, nb): return min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1.0 / nb)       # two-sided; the smallest attainable value is 1/nb
print(f"NB = {NB}: smallest attainable p = {1 / NB:.4f}; Holm over 6 tests can reach {6 / NB:.4f}")

In [ ]:
# ===================== item 2, cell 2: load every model once and run the paired bootstrap (All, PA and AP at once) =====================
MODELS = get_models()                                           # your function: CheXpert-152k and NIH-77k predictions per seed
RUNS = []
def add(kind, s, pc, pn, home): RUNS.append(dict(id=f"{kind}_s{s}", kind=kind, seed=s, home=home, pc=pc, pn=pn))
for s in SEEDS:
    add("CX152", s, MODELS["CheXpert-152k"][0][s], MODELS["CheXpert-152k"][1][s], "cx")
    add("NIH77", s, MODELS["NIH-77k"][0][s],       MODELS["NIH-77k"][1][s],       "nih")
for f in sorted(glob.glob(os.path.join(CD, f"{MODEL}_cx{NT}_e{EPOCHS}_seed*_preds.npz"))):          # EVERY random CheXpert-77k subset found on disk
    s = int(os.path.basename(f).split("seed")[-1].split("_")[0]); z = np.load(f); add("CX77", s, z["cx_test_P"], z["nih_test_P"], "cx")
print(pd.DataFrame(RUNS).groupby("kind").seed.agg(lambda x: sorted(x)).to_string())
assert sum(r["kind"] == "CX77" for r in RUNS) >= 2, "need at least 2 CheXpert-77k subsets saved in CD"

for r in RUNS:                                                  # sort every score vector once
    r["fc"] = {i: FastAUC(Yc[:, i], r["pc"][:, i]) for i in range(L)}; r["fn"] = {i: FastAUC(Ynt[:, i], r["pn"][:, i]) for i in range(L)}
vc, vn = cx_view, nih_view[off]
STRATA = {"All": (np.ones(len(Yc), bool), np.ones(len(Ynt), bool)), "PA": (vc == "PA", vn == "PA"), "AP": (vc == "AP", vn == "AP")}
LAB_USE = {"All": list(range(L)), "PA": OK["PA"], "AP": OK["AP"]}                              # findings with enough positives AND negatives in that view, in BOTH hospitals
gc_, gn_ = groups(cxt.patient_id.values), groups(nih.patient_id.values[off])                  # the resampling unit is the PATIENT
AUCB = {(r["id"], site, st): np.full((NB, L), np.nan) for r in RUNS for site in ("cx", "nih") for st in STRATA}
AUCP = {(r["id"], site, st): np.full(L, np.nan) for r in RUNS for site in ("cx", "nih") for st in STRATA}
w1c, w1n = np.ones(len(Yc)), np.ones(len(Ynt))                                                 # point estimates on the full test sets
for st, (mc_, mn_) in STRATA.items():
    for r in RUNS:
        for i in LAB_USE[st]: AUCP[(r["id"], "cx", st)][i] = r["fc"][i].auc(w1c * mc_); AUCP[(r["id"], "nih", st)][i] = r["fn"][i].auc(w1n * mn_)
rng = np.random.default_rng(2024)
for b in tqdm(range(NB), desc="paired patient bootstrap"):
    ic = np.concatenate([gc_[k] for k in rng.integers(0, len(gc_), len(gc_))]); iN = np.concatenate([gn_[k] for k in rng.integers(0, len(gn_), len(gn_))])
    wc = np.bincount(ic, minlength=len(Yc)).astype(float); wn = np.bincount(iN, minlength=len(Ynt)).astype(float)      # the SAME sampled patients for every model, view and finding
    for st, (mc_, mn_) in STRATA.items():
        wcs, wns = wc * mc_, wn * mn_
        for r in RUNS:
            for i in LAB_USE[st]: AUCB[(r["id"], "cx", st)][b, i] = r["fc"][i].auc(wcs); AUCB[(r["id"], "nih", st)][b, i] = r["fn"][i].auc(wns)
print("bootstrap done:", len(RUNS), "model runs x 3 strata x 2 hospitals x", NB, "paired resamples")

In [ ]:
# ===================== item 2, cell 3: gaps and asymmetries for any stratum =====================
RK = {k: [r for r in RUNS if r["kind"] == k] for k in ("CX152", "NIH77", "CX77")}; R = len(RK["CX77"])
macro = lambda x: np.nanmean(x, axis=-1)
def gapB(r, st): a, b = AUCB[(r["id"], "cx", st)], AUCB[(r["id"], "nih", st)]; return (a - b) if r["home"] == "cx" else (b - a)       # in-hospital minus external, (NB, L)
def gapP(r, st): a, b = AUCP[(r["id"], "cx", st)], AUCP[(r["id"], "nih", st)]; return (a - b) if r["home"] == "cx" else (b - a)       # (L,)
IDX = np.random.default_rng(7).integers(0, R, (NB, R))                                         # nested bootstrap: the R subset draws are resampled too, same draws for every quantity
dmean = lambda M: np.take_along_axis(M, IDX, 1).mean(1)
def derive(st):
    gN, gC = np.mean([gapB(r, st) for r in RK["NIH77"]], 0), np.mean([gapB(r, st) for r in RK["CX152"]], 0)           # seed-averaged, (NB, L)
    g77 = np.stack([gapB(r, st) for r in RK["CX77"]], 1)                                                              # (NB, R, L): one slice per random subset
    pN, pC = np.mean([gapP(r, st) for r in RK["NIH77"]], 0), np.mean([gapP(r, st) for r in RK["CX152"]], 0); p77 = np.stack([gapP(r, st) for r in RK["CX77"]], 0)     # (R, L)
    n77 = np.take_along_axis(g77, IDX[:, :, None], 1).mean(1)                                                         # (NB, L)
    B = {"gap NIH-77k": gN, "gap CheXpert-152k": gC, "gap CheXpert-77k": n77, "original asymmetry (NIH-77k − CheXpert-152k)": gN - gC,
         "size effect (CheXpert-77k − CheXpert-152k)": n77 - gC, "size-controlled asymmetry (NIH-77k − CheXpert-77k)": gN - n77}
    P = {"gap NIH-77k": pN, "gap CheXpert-152k": pC, "gap CheXpert-77k": p77.mean(0), "original asymmetry (NIH-77k − CheXpert-152k)": pN - pC,
         "size effect (CheXpert-77k − CheXpert-152k)": p77.mean(0) - pC, "size-controlled asymmetry (NIH-77k − CheXpert-77k)": pN - p77.mean(0)}
    return B, P, g77, p77

In [ ]:
# ===================== item 2, cell 4: PA/AP-stratified analysis, Holm per family, effect sizes =====================
D = {st: derive(st) for st in ("All", "PA", "AP")}                                              # same replicates, same patients, same subset draws for every stratum and model
rows = []
def add_row(family, quantity, st, d, est):
    d = macro(d) if d.ndim > 1 else d; lo, hi = np.percentile(d, [2.5, 97.5]); rows.append(dict(family=family, quantity=quantity, stratum=st, estimate=float(est), lo=lo, hi=hi, p=boot_p(d, NB)))
GAPQ = ["gap NIH-77k", "gap CheXpert-152k", "gap CheXpert-77k"]; ASQ = ["original asymmetry (NIH-77k − CheXpert-152k)", "size-controlled asymmetry (NIH-77k − CheXpert-77k)"]
for st in ("PA", "AP"):                                                                         # F1: is the view-matched gap different from 0?
    for q in GAPQ: add_row("F1 view-matched gaps vs 0", q, st, D[st][0][q], np.nanmean(D[st][1][q]))
for st in ("PA", "AP"):                                                                         # F2: is the asymmetry still there inside one view?
    for q in ASQ: add_row("F2 view-matched asymmetry vs 0", q, st, D[st][0][q], np.nanmean(D[st][1][q]))
for q in GAPQ + ASQ:                                                                            # F3: does the view change the gap or the asymmetry? (PA minus AP)
    add_row("F3 PA − AP contrast", q, "PA − AP", macro(D["PA"][0][q]) - macro(D["AP"][0][q]), np.nanmean(D["PA"][1][q]) - np.nanmean(D["AP"][1][q]))
S2 = pd.DataFrame(rows); S2["p_holm"] = S2.groupby("family").p.transform(lambda x: holm(x.values)); S2["significant"] = S2.p_holm < .05
print("Holm is applied inside each family (F1 6 tests, F2 4 tests, F3 5 tests); smallest attainable Holm p in F1 =", round(6 / NB, 4))
display(S2.round(4)); S2.to_csv(os.path.join(CD, "table_view_stratified_rigorous.csv"), index=False)

rows = []                                                                                       # effect size: how much of the asymmetry remains after view matching
for q in ASQ:
    for st in ("PA", "AP"):
        den = macro(D["All"][0][q]); num = macro(D[st][0][q]); ok = np.abs(den) > .01
        rat = num[ok] / den[ok]; pt = np.nanmean(D[st][1][q]) / np.nanmean(D["All"][1][q]); rows.append(dict(quantity=q, stratum=st, share_of_all_image_asymmetry_remaining=pt, lo=np.percentile(rat, 2.5), hi=np.percentile(rat, 97.5)))
EFF = pd.DataFrame(rows); display(EFF.round(3)); EFF.to_csv(os.path.join(CD, "table_view_stratified_effect_sizes.csv"), index=False)
print("A share near 1 inside BOTH views means the PA/AP mix does not explain the asymmetry; a share near 0 means it does.")

fig, axes = plt.subplots(1, 3, figsize=(COL2, 3.1))
for a_, fam_ in zip(axes, sorted(S2.family.unique())):
    g_ = S2[S2.family == fam_].reset_index(drop=True)
    for i, r in g_.iterrows():
        a_.errorbar(r.estimate, i, xerr=[[r.estimate - r.lo], [r.hi - r.estimate]], fmt="o", color="k", mfc="k" if r.significant else "w", ms=4, capsize=2, lw=.8)
    a_.axvline(0, color="r", ls="--", lw=.7); a_.set_yticks(range(len(g_))); a_.set_yticklabels([f"{q.split(' (')[0]}, {s}" for q, s in zip(g_.quantity, g_.stratum)], fontsize=5.3); a_.invert_yaxis()
    a_.set(title=fam_, xlabel="AUROC difference (95% CI)"); a_.grid(True, axis="x", alpha=.3)
fig.suptitle("Filled = significant after Holm correction within the family", fontsize=7); fig.tight_layout(); save_fig(fig, "fig_view_stratified_rigorous")

In [ ]:
# ===================== STEP 3-0: design, and VinBigData as a third, truly unseen site =====================
VBD = os.path.join(OUT, "vinbig")
vb = pd.read_csv(os.path.join(VBD, "vinbig_image_labels.csv")); NVB = len(vb)
VBA = np.memmap(os.path.join(VBD, "vb_train_224.u8"), dtype=np.uint8, mode="r", shape=(NVB, 224, 224))
Yvb = np.zeros((NVB, L), int)
for f_ in [l for l in LABELS if l != "Edema"]: Yvb[:, LABELS.index(f_)] = vb[f_].values
VALID_C = np.array([l != "Edema" for l in LABELS])                                  # VinBigData has no Edema: every VinBigData metric uses the other five findings

FRACS = [0.0, 0.25, 0.5, 0.75, 1.0]                                                 # share of the training set that comes from CheXpert
NAMEF = {0.0: "100% NIH", 0.25: "75% NIH + 25% CheXpert", 0.5: "50% NIH + 50% CheXpert", 0.75: "25% NIH + 75% CheXpert", 1.0: "100% CheXpert"}
N_TOTAL = NT                                                                        # the SAME 77,353-image budget, 1,209 steps per epoch and 7 epochs in every configuration
print(pd.DataFrame({"configuration": NAMEF.values(), "NIH images": [N_TOTAL - round(f * N_TOTAL) for f in FRACS], "CheXpert images": [round(f * N_TOTAL) for f in FRACS]}, index=[f"{int(100 * f)}% CheXpert" for f in FRACS]).to_string())
print("100% NIH = your NIH-77k models and 100% CheXpert = your CheXpert-77k models (same recipe), so only the three mixtures are trained.")

def patient_subset(pool_idx, pats, n_target, r):
    if n_target <= 0: return np.array([], dtype=int)
    if n_target >= len(pool_idx): return np.sort(pool_idx)
    cnt = pd.Series(pats).value_counts().to_dict(); up = np.unique(pats); r.shuffle(up); chosen, tot = set(), 0
    for p in up:
        chosen.add(p); tot += cnt[p]
        if tot >= n_target: break
    idx = pool_idx[np.isin(pats, list(chosen))]
    return np.sort(r.permutation(idx)[:n_target])

def mix_tag(f, seed): return f"{MODEL}_mix{int(round(100 * f)):03d}cx_n{N_TOTAL}_e{EPOCHS}_seed{seed}"

In [ ]:
# ===================== STEP 3-1: train the three mixtures (3 seeds each, about 20 min per run) =====================
# Needs cells 0, 5 and 6 of your notebook. Every run draws a NEW random patient-level subset from each source, so a seed also varies the data subset.
def train_mix(f, seed):
    tag = mix_tag(f, seed); PRD = os.path.join(CD, tag + "_preds.npz")
    if os.path.exists(PRD): print(f"{tag}: finished, skipping"); return
    BEST, CKPT, HIST = (os.path.join(CD, tag + s) for s in (".pt", "_last.pt", "_history.csv"))
    n_cx = int(round(f * N_TOTAL)); n_nih = N_TOTAL - n_cx; r = np.random.default_rng(9000 + int(1000 * f) + seed)
    ni = patient_subset(tr_i, nih.patient_id.values[tr_i], n_nih, r); ci_ = patient_subset(cx_tr_pool, cxm.patient_id.values[cx_tr_pool], n_cx, r)
    pidx = np.r_[ni, ci_]; psrc = np.r_[np.zeros(len(ni), int), np.ones(len(ci_), int)]
    print(f"{tag}: {len(ni)} NIH + {len(ci_)} CheXpert images = {len(pidx)} (patients: NIH {len(set(nih.patient_id.values[ni]))}, CheXpert {len(set(cxm.patient_id.values[ci_]))})")
    torch.manual_seed(seed); np.random.seed(seed)
    m = timm.create_model(MODEL, pretrained=True, num_classes=L).cuda(); o = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    spe = int(np.ceil(len(pidx) / BS)); sc = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=EPOCHS * spe); sl = torch.amp.GradScaler("cuda")
    hist, best, start = [], 0.0, 0
    if os.path.exists(CKPT):
        ck = torch.load(CKPT, map_location="cuda", weights_only=False)
        m.load_state_dict(ck["model"]); o.load_state_dict(ck["opt"]); sc.load_state_dict(ck["sched"]); sl.load_state_dict(ck["scaler"]); hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1
        print(f"{tag}: resumed, next epoch {start + 1}")
    for ep in range(start, EPOCHS):
        perm = np.random.default_rng(seed * 100 + ep).permutation(len(pidx)); m.train(); t0 = time.time(); rl, n = torch.zeros((), device="cuda"), 0
        for k in tqdm(range(spe), desc=f"{int(100 * f)}% CheXpert, seed {seed}, epoch {ep + 1}/{EPOCHS}"):
            sel = perm[k * BS:(k + 1) * BS]; ii, ss = pidx[sel], psrc[sel]; xs, ys = [], []
            a_, b_ = np.sort(ii[ss == 0]), np.sort(ii[ss == 1])                       # images of one batch come from both sources in proportion to the mixture
            if len(a_): xs.append(gpu_batch(arr, a_, flip=True)); ys.append(Yn[a_])
            if len(b_): xs.append(gpu_batch(CXA, b_, flip=True)); ys.append(Ycm[b_])
            x = torch.cat(xs); y = torch.from_numpy(np.concatenate(ys)).cuda().float()
            o.zero_grad(set_to_none=True)
            with torch.autocast("cuda"): loss = lossf(m(x), y)
            sl.scale(loss).backward(); sl.step(o); sl.update(); sc.step(); rl += loss.detach() * len(x); n += len(x)
        vals = []                                                                     # model selection on the validation sets of the sources that are present, never on a test set
        if len(ni): vals.append(mean_auc(Yn[va_i], predict_idx(m, arr, va_i)))
        if len(ci_): vals.append(mean_auc(Ycm[cx_va], predict_idx(m, CXA, cx_va)))
        vm = float(np.mean(vals)); row = dict(epoch=ep + 1, train_loss=float(rl) / n, val_auroc=vm, sec=time.time() - t0); hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"{tag} ep {ep + 1} | loss {row['train_loss']:.4f} | val AUROC {vm:.4f} | {row['sec'] / 60:.1f} min")
        if vm > best: best = vm; torch.save(m.state_dict(), BEST)
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=sc.state_dict(), scaler=sl.state_dict(), hist=hist, best=best, ep=ep), CKPT)
    m.load_state_dict(torch.load(BEST, map_location="cuda", weights_only=True))
    z = dict(cx_test_P=predict_idx(m, XC, np.arange(len(XC))), nih_test_P=predict_idx(m, arr, te_i), vb_P=predict_idx(m, VBA, np.arange(NVB)))
    np.savez(PRD, **z); print(f"{tag}: CheXpert test {mean_auc(Yc, z['cx_test_P']):.4f} | NIH test {mean_auc(Ynt, z['nih_test_P']):.4f}")
    del m, o, sc, sl; gc.collect(); torch.cuda.empty_cache()

for f in (0.25, 0.5, 0.75):
    for s in SEEDS: train_mix(f, s)

In [ ]:
# ===================== STEP 3-2: VinBigData predictions for the two endpoint configurations (from your existing checkpoints) =====================
def endpoint_ckpt(f, s):                                                              # edit here if your checkpoint names differ
    return os.path.join(RD, f"{MODEL}_nih_e{EPOCHS}_seed{s}.pt") if f == 0.0 else os.path.join(CD, f"{MODEL}_cx{NT}_e{EPOCHS}_seed{s}.pt")
def endpoint_vb(f, s):
    out = os.path.join(CD, f"{MODEL}_endpoint_f{int(100 * f):03d}_seed{s}_vb.npy")
    if os.path.exists(out): return np.load(out)
    ck = endpoint_ckpt(f, s); assert os.path.exists(ck), f"missing checkpoint {ck}: edit endpoint_ckpt"
    m = timm.create_model(MODEL, pretrained=False, num_classes=L).cuda(); m.load_state_dict(torch.load(ck, map_location="cuda", weights_only=True))
    P = predict_idx(m, VBA, np.arange(NVB)); np.save(out, P); del m; gc.collect(); torch.cuda.empty_cache(); return P
for f in (0.0, 1.0):
    for s in SEEDS: endpoint_vb(f, s)
print("endpoint predictions on VinBigData ready")

In [ ]:
# ===================== STEP 3-3: load all 15 models and run ONE paired bootstrap over test patients =====================
M2 = get_models(); CMP = []
for s in SEEDS:
    CMP.append(dict(f=0.0, seed=s, pc=M2["NIH-77k"][0][s], pn=M2["NIH-77k"][1][s], pv=endpoint_vb(0.0, s)))
    z = np.load(os.path.join(CD, f"{MODEL}_cx{NT}_e{EPOCHS}_seed{s}_preds.npz")); CMP.append(dict(f=1.0, seed=s, pc=z["cx_test_P"], pn=z["nih_test_P"], pv=endpoint_vb(1.0, s)))
    for f in (0.25, 0.5, 0.75):
        z = np.load(os.path.join(CD, mix_tag(f, s) + "_preds.npz")); CMP.append(dict(f=f, seed=s, pc=z["cx_test_P"], pn=z["nih_test_P"], pv=z["vb_P"]))
for k, r in enumerate(CMP):
    r["id"] = k; r["fc"] = {i: FastAUC(Yc[:, i], r["pc"][:, i]) for i in range(L)}; r["fn"] = {i: FastAUC(Ynt[:, i], r["pn"][:, i]) for i in range(L)}
    r["fv"] = {i: FastAUC(Yvb[:, i], r["pv"][:, i]) for i in range(L) if VALID_C[i]}
SITES = ("cx", "nih", "vb"); SITE_LAB = {"cx": "CheXpert test", "nih": "NIH test", "vb": "VinBigData (unseen, 5 findings)"}
gc_, gn_ = groups(cxt.patient_id.values), groups(nih.patient_id.values[off])           # patient is the resampling unit; VinBigData has no patient id, so its unit is the image
AUC3 = {(r["id"], s_): np.full((NB, L), np.nan) for r in CMP for s_ in SITES}; PT3 = {(r["id"], s_): np.full(L, np.nan) for r in CMP for s_ in SITES}
ones = {"cx": np.ones(len(Yc)), "nih": np.ones(len(Ynt)), "vb": np.ones(NVB)}
for r in CMP:
    for i in range(L):
        PT3[(r["id"], "cx")][i] = r["fc"][i].auc(ones["cx"]); PT3[(r["id"], "nih")][i] = r["fn"][i].auc(ones["nih"])
        if VALID_C[i]: PT3[(r["id"], "vb")][i] = r["fv"][i].auc(ones["vb"])
rng = np.random.default_rng(303)
for b in tqdm(range(NB), desc="paired bootstrap, all configurations"):
    ic = np.concatenate([gc_[k] for k in rng.integers(0, len(gc_), len(gc_))]); iN = np.concatenate([gn_[k] for k in rng.integers(0, len(gn_), len(gn_))]); iV = rng.integers(0, NVB, NVB)
    w = {"cx": np.bincount(ic, minlength=len(Yc)).astype(float), "nih": np.bincount(iN, minlength=len(Ynt)).astype(float), "vb": np.bincount(iV, minlength=NVB).astype(float)}   # SAME sampled patients for every model
    for r in CMP:
        for i in range(L):
            AUC3[(r["id"], "cx")][b, i] = r["fc"][i].auc(w["cx"]); AUC3[(r["id"], "nih")][b, i] = r["fn"][i].auc(w["nih"])
            if VALID_C[i]: AUC3[(r["id"], "vb")][b, i] = r["fv"][i].auc(w["vb"])
print("done:", len(CMP), "models x 3 test sets x", NB, "paired resamples")

In [ ]:
 # ===================== STEP 3-4: tables, pre-specified contrasts (Holm per family), trend slopes =====================
S = len(SEEDS); rng2 = np.random.default_rng(404)
RF_ = {f: [r for r in CMP if r["f"] == f] for f in FRACS}
macro = lambda x: np.nanmean(x, axis=-1)
MAC = {(f, s_): np.stack([macro(AUC3[(r["id"], s_)]) for r in RF_[f]], 1) for f in FRACS for s_ in SITES}          # (NB, S): macro AUROC per replicate and per seed
MACP = {(f, s_): np.array([np.nanmean(PT3[(r["id"], s_)]) for r in RF_[f]]) for f in FRACS for s_ in SITES}          # (S,)
IDXS = {f: rng2.integers(0, S, (NB, S)) for f in FRACS}                                                                # seeds (= random data subsets) are resampled too, independently per configuration
BM = {(f, s_): np.take_along_axis(MAC[(f, s_)], IDXS[f], 1).mean(1) for f in FRACS for s_ in SITES}                    # (NB,)
PM_ = {(f, s_): MACP[(f, s_)].mean() for f in FRACS for s_ in SITES}
for f in FRACS:
    BM[(f, "seen_mean")] = (BM[(f, "cx")] + BM[(f, "nih")]) / 2; PM_[(f, "seen_mean")] = (PM_[(f, "cx")] + PM_[(f, "nih")]) / 2
    BM[(f, "worst")] = np.minimum(BM[(f, "cx")], BM[(f, "nih")]); PM_[(f, "worst")] = min(PM_[(f, "cx")], PM_[(f, "nih")])      # worst of the two hospitals

rows = []
for f in FRACS:
    r_ = {"configuration": NAMEF[f]}
    for s_ in SITES: r_[SITE_LAB[s_]] = f"{MACP[(f, s_)].mean():.3f} ± {MACP[(f, s_)].std(ddof=1):.3f}"
    r_["mean of NIH and CheXpert"] = f"{PM_[(f, 'seen_mean')]:.3f}"; r_["worst of NIH and CheXpert"] = f"{PM_[(f, 'worst')]:.3f}"; rows.append(r_)
T3 = pd.DataFrame(rows).set_index("configuration"); display(T3); T3.to_csv(os.path.join(CD, "table_composition_auroc.csv"))
print("mean ± sd over seeds (each seed = a new random data subset); the contrasts below use the paired patient bootstrap and also resample the seeds.")

crow = []
def contrast(fam, f1, f0, key, label):
    d = BM[(f1, key)] - BM[(f0, key)]; lo, hi = np.percentile(d, [2.5, 97.5]); crow.append(dict(family=fam, contrast=label, estimate=PM_[(f1, key)] - PM_[(f0, key)], lo=lo, hi=hi, p=boot_p(d, NB)))
for f in (.25, .5, .75, 1.0): contrast("F1 NIH test: vs 100% NIH", f, 0.0, "nih", f"{NAMEF[f]} − 100% NIH")
for f in (0.0, .25, .5, .75): contrast("F2 CheXpert test: vs 100% CheXpert", f, 1.0, "cx", f"{NAMEF[f]} − 100% CheXpert")
for f in (0.0, .25, .5, .75): contrast("F3 VinBigData (unseen): vs 100% CheXpert", f, 1.0, "vb", f"{NAMEF[f]} − 100% CheXpert")     # reference fixed in advance: CheXpert was the more transferable source in the earlier analysis
for f in (.25, .5, .75):
    for f0 in (0.0, 1.0): contrast("F4 mean of NIH and CheXpert: mixture vs each single source", f, f0, "seen_mean", f"{NAMEF[f]} − {NAMEF[f0]}")
x = np.array(FRACS); xc = x - x.mean()
def slope_of(vals): return (np.asarray(vals) * xc).sum(-1) / (xc ** 2).sum() * 0.25                                          # AUROC change per +25 percentage points of CheXpert
for key, nm in [("nih", "NIH test"), ("cx", "CheXpert test"), ("vb", "VinBigData (unseen)"), ("seen_mean", "mean of NIH and CheXpert"), ("worst", "worst of NIH and CheXpert")]:
    d = slope_of(np.stack([BM[(f, key)] for f in FRACS], 1)); lo, hi = np.percentile(d, [2.5, 97.5])
    crow.append(dict(family="F5 trend: AUROC change per +25 points of CheXpert", contrast=nm, estimate=float(slope_of([PM_[(f, key)] for f in FRACS])), lo=lo, hi=hi, p=boot_p(d, NB)))
C3 = pd.DataFrame(crow); C3["p_holm"] = C3.groupby("family").p.transform(lambda v: holm(v.values)); C3["significant"] = C3.p_holm < .05
print(f"Holm is applied inside each family; smallest attainable raw p = {1 / NB:.4f}")
display(C3.round(4)); C3.to_csv(os.path.join(CD, "table_composition_contrasts.csv"), index=False)

In [ ]:
# ===================== STEP 3-5: figures =====================
xs = [100 * f for f in FRACS]; col = {"nih": "0.15", "cx": "0.5", "vb": "0.8"}; mk = {"nih": "o", "cx": "s", "vb": "^"}
fig, ax = plt.subplots(1, 2, figsize=(COL2, 3.0))
for key in ("nih", "cx", "vb"):
    mean = [PM_[(f, key)] for f in FRACS]; lo = [np.percentile(BM[(f, key)], 2.5) for f in FRACS]; hi = [np.percentile(BM[(f, key)], 97.5) for f in FRACS]
    ax[0].plot(xs, mean, marker=mk[key], color=col[key], mec="k", mew=.5, label=SITE_LAB[key]); ax[0].fill_between(xs, lo, hi, color=col[key], alpha=.25, lw=0)
    for f in FRACS: ax[0].scatter([100 * f] * S, MACP[(f, key)], s=5, color="k", alpha=.35, zorder=3)
ax[0].set(xlabel="% of training images from CheXpert", ylabel="Macro AUROC", title="Test hospitals and an unseen hospital"); ax[0].legend(frameon=False, fontsize=6); ax[0].grid(True, alpha=.3)
for key, nm, c in [("seen_mean", "mean of NIH and CheXpert test", "0.3"), ("worst", "worst of NIH and CheXpert test", "0.7")]:
    mean = [PM_[(f, key)] for f in FRACS]; lo = [np.percentile(BM[(f, key)], 2.5) for f in FRACS]; hi = [np.percentile(BM[(f, key)], 97.5) for f in FRACS]
    ax[1].plot(xs, mean, marker="o", color=c, mec="k", mew=.5, label=nm); ax[1].fill_between(xs, lo, hi, color=c, alpha=.25, lw=0)
ax[1].set(xlabel="% of training images from CheXpert", title="Does mixing help the weaker hospital?"); ax[1].legend(frameon=False, fontsize=6); ax[1].grid(True, alpha=.3)
fig.suptitle("Fixed training budget of 77,353 images; points = single seeds, bands = 95% paired-bootstrap CI", fontsize=7); fig.tight_layout(); save_fig(fig, "fig_composition_curves")

fams = sorted(C3.family.unique()); fig, axes = plt.subplots(1, len(fams), figsize=(COL2 * 1.5, 3.4))
for a_, fam_ in zip(axes, fams):
    g_ = C3[C3.family == fam_].reset_index(drop=True)
    for i, r in g_.iterrows(): a_.errorbar(r.estimate, i, xerr=[[r.estimate - r.lo], [r.hi - r.estimate]], fmt="o", color="k", mfc="k" if r.significant else "w", ms=4, capsize=2, lw=.8)
    a_.axvline(0, color="r", ls="--", lw=.7); a_.set_yticks(range(len(g_))); a_.set_yticklabels(g_.contrast, fontsize=4.8); a_.invert_yaxis(); a_.set_title(fam_.split(":")[0], fontsize=7); a_.set_xlabel("AUROC difference (95% CI)", fontsize=6); a_.grid(True, axis="x", alpha=.3)
fig.suptitle("Filled = significant after Holm correction within the family", fontsize=7); fig.tight_layout(); save_fig(fig, "fig_composition_contrasts")